Ines Harraoui, 2120496
Loïc Huang, 20213839

**Which LLM(s) did you use for this work?**
Perplexity, ChatGPT

# PROGRES 2023 - Mini-Projet 2
# API Web

Fabien Mathieu - fabien.mathieu@lip6.fr

Sébastien Tixeuil - Sebastien.Tixeuil@lip6.fr

The purpose of this mini-project is to work with the *Internet Movie DataBase* (IMDB) and a Python Web framework. It will involve:

- Retrieve and manipulate datasets
- Build an API to perform various tasks on the data
- Build a website that will use the API above

# Rules

1. Cite your sources
2. One file to rule them all
3. Explain
4. Execute your code


https://github.com/balouf/progres/blob/main/rules.ipynb

# The IMDB dataset

[IMDB](https://www.imdb.com) allows to retrieve a part of its dataset for any non-commercial purpose. The available data and the formatting convention is described here: https://developer.imdb.com/non-commercial-datasets/

We are especially interested in the data from the following files:
- https://datasets.imdbws.com/title.principals.tsv.gz
- https://datasets.imdbws.com/name.basics.tsv.gz
- https://datasets.imdbws.com/title.basics.tsv.gz

**Important notes**:
- If you see *Your answer here*, that means something is expected from you.
- To help you, the start and/or the end of a possible solution is sometimes given.
- The content of IMDB is refreshed regularly. That means that some of the results you will compute, like the number of movies, will vary with time. This should not surprise you.

## Exercise 1: Download

Write a `download_imdb` function inspired by the `download` function seen in course, with the following modifications:
- `download_imdb` will have one single argument, the name of the file to retrieve. Server location of the file is assumed to be https://datasets.imdbws.com/
- If the file already exists, print a message telling that it exists and do nothing. You can use the `pathlib` module for that.
 The data files are quite big, so you will specify a directory `data_dir` where the data files will be stored/read.

Your answer here.

Prompts : ChatGPT
> Contexte:
> 
>2 variables à disposition : 
>- data_dir : L'endroit où le fichier doit être téléchargé s'il n'est pas présent dans le repertoire
>- base_url : Contient l'URL du site où on doit récupérer le fichier
>
>2 Bibliothèque : 
>- Pathlib qui import path
>- requests qui import Session
>
>Génère moi une fonction download_imdb qui prend en paramètre un argument : le nom d'un fichier, et qui doit me télécharger le fichier passé en paramètre. Cependant, si le fichier est déjà dans mon repertoire, on affichera juste que le fichier "nom" existe déjà. Explique moi comme un étudiant en université, de manière à ce que je comprends ta manière de procéder. Commence d'abord par expliquer ce que ton code fait. Ensuite, explique la logique derrière chaque boucle, condition etc... Tu peux faire un exemple pour illustrer ce que tu expliques.

"La fonction download_imdb(fichier) a pour objectif de :
- Vérifier si le fichier existe déjà dans un dossier local (data_dir).
- Si le fichier existe, afficher un message indiquant qu’il existe déjà.
- Sinon, le télécharger depuis l’URL de base (base_url) combinée au nom du fichier et le sauvegarder dans data_dir."
  
Pour faire court, la fonction télécharge un fichier dans le répertoire seulement si le fichier n'existe pas.

In [64]:
from pathlib import Path
from requests import Session

base_url = "https://datasets.imdbws.com/"
data_dir = Path.home() / "Downloads"

def download_imdb(file):

    dest_file = Path(data_dir) / file
    if dest_file.exists():
        print(f"Le fichier '{file}' existe déjà.")
        return

    source_url =  base_url + file
        
    dest_file.parent.mkdir(parents=True, exist_ok=True)
    
    s = Session()
    r = s.get(source_url, stream=True)
    
    with open(dest_file, "wb") as f:
        for chunk in r.iter_content(chunk_size=8192):
            if chunk:
                f.write(chunk)


In [65]:
files = ['title.principals.tsv.gz', 'name.basics.tsv.gz', 'title.basics.tsv.gz']
for file in files:
    download_imdb(file)

Le fichier 'title.principals.tsv.gz' existe déjà.
Le fichier 'name.basics.tsv.gz' existe déjà.
Le fichier 'title.basics.tsv.gz' existe déjà.


## Exercise 2: Explore

- What is the size of the different files you retrieved? You can use Python or a file explorer, as you prefer.

Your answer here.

Prompts : ChatGPT
> Peux-tu m'expliquer comment on peut avoir la taille d'un fichier ?

> Pourquoi sur mon explorateur de fichier je vois 744.5 Mo pour un fichier mais dans mes print j'ai 720Mo ?

ChatGPT m'a expliqué qu'il avait une différence entre la taille que python voyait et la taille de l'explorateur de fichier : "Python utilise la base 2, que les informaticiens appellent souvent “Mibibytes (MiB)” pour être précis. L’explorateur de fichiers utilise la base 10, les “Mégaoctets (MB)” du marketing." C'est pourquoi il est préférable d'utiliser / 1000000 plutôt que / (1024*1024) si on veut avoir le même affichage que celui de l'explorateur de fichier


In [66]:
from pathlib import Path

for file in files:
    fichier = Path(file)
    taille = fichier.stat().st_size
    taille_mo = taille/ 1000000
    print(f"Le fichier {fichier} a une taille de {taille_mo:.2f} Mo")


Le fichier title.principals.tsv.gz a une taille de 746.29 Mo
Le fichier name.basics.tsv.gz a une taille de 296.27 Mo
Le fichier title.basics.tsv.gz a une taille de 215.48 Mo


As explained in https://developer.imdb.com/non-commercial-datasets/:
- the data is stored as `tsv`, which means each text line represents a row.
- A [gzip compression](https://docs.python.org/3/library/gzip.html) is used to reduce the size of the data on the hard drive.

Large compressed files should not be uncompressed on your hard drive or fully loaded in memory.

The Python [gzip module](https://docs.python.org/3/library/gzip.html) is designed so you can open a compressed file as if it was already uncompressed. For example, the following code reads 666 lines from `title.basics` and print the last line read.

In [67]:
import gzip
with gzip.open(data_dir / 'title.basics.tsv.gz', 'rt', encoding='utf8') as f:
    for _ in range(666):
        l = f.readline()
print(l)

tt0000671	short	Desdemona	Desdemona	0	1908	\N	\N	Drama,Short



- Write a function that read the 4 first lines of a compressed tsv file. Each line read should be converted into a list of elements and printed.

Your answer here.

Source : Diapo de cours

In [68]:
def explore(name):
    with gzip.open(data_dir / name, 'rt', encoding='utf8') as f:
        for _ in range(4):
            l = f.readline()
            l = l.strip().split("\t")
            print(l)
        

In [69]:
for file in files:
    print(f"First lines of {file}:")
    explore(file)

First lines of title.principals.tsv.gz:
['tconst', 'ordering', 'nconst', 'category', 'job', 'characters']
['tt0000001', '1', 'nm1588970', 'self', '\\N', '["Self"]']
['tt0000001', '2', 'nm0005690', 'director', '\\N', '\\N']
['tt0000001', '3', 'nm0005690', 'producer', 'producer', '\\N']
First lines of name.basics.tsv.gz:
['nconst', 'primaryName', 'birthYear', 'deathYear', 'primaryProfession', 'knownForTitles']
['nm0000001', 'Fred Astaire', '1899', '1987', 'actor,miscellaneous,producer', 'tt0072308,tt0050419,tt0027125,tt0025164']
['nm0000002', 'Lauren Bacall', '1924', '2014', 'actress,miscellaneous,soundtrack', 'tt0037382,tt0075213,tt0038355,tt0117057']
['nm0000003', 'Brigitte Bardot', '1934', '\\N', 'actress,music_department,producer', 'tt0057345,tt0049189,tt0056404,tt0054452']
First lines of title.basics.tsv.gz:
['tconst', 'titleType', 'primaryTitle', 'originalTitle', 'isAdult', 'startYear', 'endYear', 'runtimeMinutes', 'genres']
['tt0000001', 'short', 'Carmencita', 'Carmencita', '0', '

- How many movie entries are present in the retrieved database?
- How many people entries?

Your answer here.

Prompt : ChatGPT
> Tu penses qu'il y a une meilleure façon de faire ce code ? (Insérer le code), on m'a déjà expliqué que lorsqu'on fait f.readlines() ça stockait tout en mémoire. Je viens de voir qu'on a plus de 1 million de lignes à chaque fois...

ChatGPT m'a expliqué qu'il était possible de compter ligne par ligne sans charger tout en mémoire mais qu'il était également possible de faire une fonction générique qui compte pour n'importe quel fichier au lieu de devoir écrire le même code en changeant juste le nom du fichier. On peut également utiliser la bibliothèque pandas pour qu'il soit encore plus rapide. 
Les avantages avec le code maintenant : 
- "Très peu de mémoire utilisée, même pour des fichiers de plusieurs Go."
- "Simple et lisible."
- "Facile à réutiliser pour d’autres fichiers IMDb."

In [70]:
def compter_lignes_gzip(filepath):
    with gzip.open(filepath, 'rt', encoding='utf8') as f:
        return sum(1 for _ in f) - 1

films = compter_lignes_gzip(data_dir / 'title.basics.tsv.gz')
personnes = compter_lignes_gzip(data_dir / 'name.basics.tsv.gz')

print(f"Il y a en tout {films} films")
print(f"Au total, nous avons {personnes} personnes enregistrées")

Il y a en tout 12171440 films
Au total, nous avons 14968592 personnes enregistrées


## Exercise 3: Extract

We want to study the relations between actors and movies. In particular, we focus on:
- Actual movies (e.g. not TV shows or short movies), where the movie year is known and at least one actor/actress is credited.
- Actors that are credited in at least one actual movie.

To start with, build a [Python set](https://docs.python.org/3/tutorial/datastructures.html#sets) that contains all movie ids (`tconst`) such that:
- The type of movie (`titleType`) is `movie`;
- The year (`startYear`) exists, i.e. is an integer.

How many movies have you referenced in the set?

Your answer here.

Prompt: ChatGPT
> Pourquoi avec mon code, le résultat que j'ai n'est pas le même que le resultat attendu ? J'ai un resultat de 625809 a la place de 624797

D'après ChatGPT, il est possible que la version du fichier IMDb soit plus récente, c'est pourquoi la valeur n'est pas la même. 

In [71]:
true_movies = set()

with gzip.open(data_dir / 'title.basics.tsv.gz', 'rt', encoding='utf8') as f:
    next(f)
    for l in f:
        l = l.strip().split("\t")
        if (l[1] == 'movie') and (l[5] != '\\N') and (l[5].isdigit()):
            true_movies.add(l[0])


In [72]:
len(true_movies)

626404

Now we want to build two lists, `movies` and `actors`:

- Each element of `movies` should represent a movie, each element of `actors` an actor or actress;
- A movie is represented by a list of three elements:
  - The original name of the movie (`str`),
  - The principal actors of the movie, stored as a list whose elements are integers that represent the index (position) of the actors in the list `actors`,
  - The movie year, `startYear` (`int`);
- An actor/actress is represented by a list of two elements:
  - The name of the person (`str`),
  - The movies the person acted in, stored as a list whose elements are integers that represent the index (position) of the movies in the list `movies`.
  

Build these two lists.

A possible way to do this (this is a suggestion, not an order):
- Initiate `movies` and `actors` as empty lists;
- Create two auxiliary dictionary that will associate to each movie id (`tconst`) and person id (`nconst`) their position in the list;
- Read the file `title.principals.tsv.gz` line by line:
  - Ignore any line where the movie is not in the set `true_movies` or the `category` of the relation is not `actor` or `actress`,
  - If the movie id `tconst` is not in the movie auxiliary index, append an empty movie to `movies` (`["", [], 0]`) and update the movie auxiliary index with an entry for `tconst`,
  - If the actor id `nconst` is not in the actor auxiliary index, append an empty actor to `actors` (`["", []]`) and update the actor auxiliary index with an entry for `nconst`,
  - Append the movie index (not `tconst`!) to the movies of the corresponding actor in `actors`,
  - Append the actor index (not `nconst`!) to the actors of the corresponding movie in `movies`;
- There can be a few undesired duplicates, e.g. some actors can have multiple entries for the same movies. For each actor, remove possible duplicates in the list of movies, and for each movie, remove possible duplicates in the list of actors;
- Using `title.basics.tsv.gz` and your movie auxiliary index, populate each movie in `movies` with its correct name (`str`) and year (`int`);
- Using `name.basics.tsv.gz` and your actor auxiliary index, populate each actor in `movies` with her correct name.

Your answer here.

Prompts:
> Peux-tu me réexpliquer les étapes ? Sans me donner le code (Insere la consigne)

> Est-ce normal que mon code prenne beaucoup de temps à s'exécuter ? (Inserer le code) A chaque execution, je dois attendre presque plusieurs minutes pour que ça s'exécute.

Les raisons pour lesquelles le temps d'exécution est grand sont les tailles des fichiers : "title.principals.tsv.gz fait souvent plusieurs centaines de Mo. title.basics.tsv.gz et name.basics.tsv.gz sont aussi très volumineux. Le code lit chaque ligne et effectue plusieurs opérations pour chaque ligne : vérification dans des sets/dictionnaires, mise à jour des listes, recherche d’indices… Même si chaque opération individuelle est rapide, le nombre total de lignes multiplié par ces opérations fait que ça prend du temps."

On pourrait utiliser des sets à la place des listes pour l'instant puis les retransformer en liste plus tard, pour gagner en temps.  Mais l'éxecution prend quand même du temps. Il faut compter entre 1 à 2 minutes.

In [73]:
movie_id_to_index = dict()
movies = []
actor_id_to_index = dict()
actors = []

with gzip.open(data_dir / 'title.principals.tsv.gz', 'rt', encoding='utf8') as f:
    next(f)
    i = 0
    j = 0
    for l in f:
        l = l.strip().split("\t")
        if (l[0] in true_movies) and ((l[3] == "actor") or (l[3] == "actress")): 
            if movie_id_to_index.get(l[0]) is None:
                movie_id_to_index[l[0]] = i
                i += 1
                movies.append(["",[],0])
            if actor_id_to_index.get(l[2]) is None:
                actor_id_to_index[l[2]] = j
                j += 1
                actors.append(["",[]])
            res_movie = movie_id_to_index.get(l[0])
            res_actor = actor_id_to_index.get(l[2])
            liste_movie = movies[res_movie]
            liste_actor = actors[res_actor]
            if res_actor not in liste_movie[1]:
                liste_movie[1].append(res_actor)
            if res_movie not in liste_actor[1]:
                liste_actor[1].append(res_movie)
            
with gzip.open(data_dir / 'title.basics.tsv.gz', 'rt', encoding='utf8') as f:
    next(f)
    for l in f:
        l = l.strip().split("\t")
        indice = movie_id_to_index.get(l[0])
        if indice is not None:
            movies[indice][0] = l[3]
            movies[indice][2] = int(l[5])
            
with gzip.open(data_dir / 'name.basics.tsv.gz', 'rt', encoding='utf8') as f:
    next(f)
    for l in f:
        l = l.strip().split("\t")
        indice = actor_id_to_index.get(l[0])
        if indice is not None:
            actors[indice][0] = l[1]

            

print("Ellipsis")

Ellipsis


Manually check that your files are correct. For example, try to get the name and year of the movies Michel Blanc played in, or the actors of the first Harry Potter movie.

Your answer here (if everything went well, you just need to execute the two cells below).

In [74]:
', '.join([f"{movies[i][0]} ({movies[i][2]})" for i in [a for a in actors if a[0]=='Michel Blanc'][0][1]])

"La meilleure façon de marcher (1976), Vous n'aurez pas l'Alsace et la Lorraine (1977), Les bronzés (1978), Les bronzés font du ski (1979), Cause toujours... tu m'intéresses! (1979), Le cheval d'orgueil (1980), La gueule de l'autre (1979), Ma femme s'appelle reviens (1982), Viens chez moi, j'habite chez une copine (1981), Le père Noël est une ordure (1982), Circulez y a rien à voir! (1983), Papy fait de la résistance (1983), Retenez-moi... ou je fais un malheur! (1984), Marche à l'ombre (1984), Nemo (1984), Drôle de samedi (1985), Je hais les acteurs (1986), Tenue de soirée (1986), Une nuit à l'Assemblée Nationale (1988), Monsieur Hire (1989), Chambre à part (1989), Uranus (1990), The Favour, the Watch and the Very Big Fish (1991), Merci la vie (1991), Prospero's Books (1991), Toxic Affair (1993), Grosse fatigue (1994), Il mostro (1994), Les grands ducs (1996), Rien ne va plus (1979), Le beaujolais nouveau est arrivé (1978), Embrassez qui vous voudrez (2002), Madame Edouard (2004), Les

In [75]:
', '.join([actors[i][0] for i in [m for m in movies if m[0].startswith('Harry Potter')][0][1]])

'Daniel Radcliffe, Rupert Grint, Emma Watson, Richard Harris, Maggie Smith, Robbie Coltrane, Saunders Triplets, Fiona Shaw, Harry Melling, Richard Griffiths'

When you have successfully reached this point of the project, you can save the two lists `movies` and `actors` as compressed json files using the code below:

In [76]:
import gzip
import json

with gzip.open(data_dir / 'movies.json.gz', 'wt', encoding='utf8') as f:
    json.dump(movies, f)
with gzip.open(data_dir / 'actors.json.gz', 'wt', encoding='utf8') as f:
    json.dump(actors, f)

After your files have been saved, you do not need to re-execute all of the above each time your restart your notebook. Instead, you just need to reload `movies` and `actors` using the code below:

In [77]:
import gzip
import json

with gzip.open(data_dir / 'movies.json.gz', 'rt', encoding='utf8') as f:
    movies = json.load(f)
with gzip.open(data_dir / 'actors.json.gz', 'rt', encoding='utf8') as f:
    actors = json.load(f)    

**Important remark:** in what follows, you will have to build functions that use the two lists a lot. You should NOT reload the lists each time you call a function. Instead, ensure that the two lists are loaded in memory and use them directly.

## Exercise 4: Explore again (now on the curated dataset)

- How many actors do you have in the new dataset? How many movies?
- In average, in how many movies played an actor?
- In average, how many actors play in a movie?
- What is the name of the actor that played in the most movies? How many movies did he feature in?
- What is the oldest movie in the DB?

Your answer here.


In [78]:
#Nombre d'entrées de chaque database
print(f"Il y a {len(movies)} films")
print(f"Il y a {len(actors)} acteurs")
print("-----------------------------")

#Moyenne 
somme_acteur = 0
somme_film = 0
for liste_a in movies:
    somme_acteur += len(liste_a[1])
for liste_m in actors:
    somme_film += len(liste_m[1])

print(f"En moyenne il y a {somme_acteur//len(movies)} acteurs par film")
print(f"En moyenne il y a {somme_film//len(actors)} films par acteurs")
print("-----------------------------")

#L'acteur avec le plus de film joué
max_acteur = None
max_films = 0

for acteur in actors:
    if len(acteur[1]) > max_films:
        max_films = len(acteur[1])
        max_acteur = acteur[0]
print(f"L'acteur qui a jouer le plus de film est : {max_acteur}")
print("-----------------------------")

#Le film le plus ancien 
nom_film = movies[0][0]
vieux_film = movies[0][2]
for film in movies:
    if vieux_film > film[2]:
        nom_film = film[0]
        vieux_film = film[2]
print(f"Le film le plus ancien est : {nom_film}")


Il y a 492714 films
Il y a 1264225 acteurs
-----------------------------
En moyenne il y a 7 acteurs par film
En moyenne il y a 3 films par acteurs
-----------------------------
L'acteur qui a jouer le plus de film est : Brahmanandam
-----------------------------
Le film le plus ancien est : Miss Jerry


## Exercise 5: Prepare some functions

Write the following functions
- `search_movie(name: str) -> list`: return a list of movies whose name contains `name` (ignoring case). Each movie is described as a dictionary with keys `name`, `year`, and `index` (its position in `movies`)
- `get_movie(i: int) -> dict`: returns the a json of the movie at position `i`, with following keys:
  - `name` (`str`)
  - `year` (`int`)
  - `actors` (list of dictionaries with keys `name` and `index`)
- `search_actor(name: str) -> list`: return a list of actors whose name contains `name` (ignoring case). Each actor is described as a dictionary with keys `name` and `index` (its position in `actor`)
- `get_actor(i: int) -> dict`: returns the a json of the actor at position `i`, with following keys:
  - `name` (`str`)
  - `movies` (list of dictionaries with keys `name`, `year`, and `index`)

Your answer here.

Prompts :
> En python, comment on fait pour vérifier que notre chaîne de caractères contient un pattern ? Par exemple j'ai la chaine "Comment ça va ?" je veux vérifier qu'il y a bien "ça" dans ce texte. Je pensais utiliser les expressions regulières.

> Après la commande %%writefile MP2_fun.py, est ce que les fonctions définies juste après sont quand même définies dans la mémoire ? Parce que les fonctions sont dans des cellules plus bas. Tout est exécuté dans un notebook.

> (Insérer le code) Explique le code comme à un étudiant en université. Commence d'abord par expliquer ce que le code fait. Ensuite, explique la logique derrière chaque boucle, condition etc... Tu peux faire un exemple pour illustrer ce que tu expliques.

Nous avons 4 fonctions qui utilisent 2 listes :
- "movies : chaque élément est un film avec son nom, son année et la liste des indices des acteurs principaux."
- "actors : chaque élément est un acteur avec son nom et la liste des indices des films dans lesquels il a joué."

"Les fonctions permettent de : 
1. Chercher des films ou acteurs par nom : 
- search_movie(name) → retourne tous les films dont le nom contient name.
- search_actor(name) → retourne tous les acteurs dont le nom contient name.
2. Obtenir les détails d’un film ou d’un acteur à partir de son indice :
- get_movie(i) → retourne le nom, l’année et les acteurs d’un film.
- get_actor(i) → retourne le nom et les films d’un acteur.

Les fonctions utilisent ces indices pour naviguer facilement entre les deux listes et construire des dictionnaires lisibles."

In [79]:
import re
def search_movie(name):
    resultat = []
    for i in range(len(movies)):
        if re.search(name.lower(), (movies[i][0]).lower()):
            val = dict()
            val['name'] = movies[i][0]
            val['year'] = movies[i][2]
            val['index'] = i
            resultat.append(val)
    return resultat
        

In [80]:
def get_movie(i):
    resultat = dict()
    if len(movies) > i: 
        resultat['name'] = movies[i][0]
        resultat['year'] = movies[i][2]
        liste_acteur = []
        for indice in movies[i][1]:
            val = dict()
            val['name'] = actors[indice][0]
            val['index'] = indice
            liste_acteur.append(val)
        resultat['actor'] = liste_acteur
    return resultat

In [81]:
def search_actor(name):
    resultat = []
    for i in range(len(actors)):
        if re.search(name.lower(), (actors[i][0]).lower()):
            val = dict()
            val['name'] = actors[i][0]
            val['index'] = i
            resultat.append(val)
    return resultat

In [82]:
def get_actor(i):
    resultat = dict()
    if len(actors) > i: 
        resultat = dict()
        resultat['name'] = actors[i][0]
        liste_film = []
        for indice in actors[i][1]:
            film = {
                "name": movies[indice][0],
                "year": movies[indice][2],
                "index": indice
            }
            liste_film.append(film)
        resultat['movies'] = liste_film
    return resultat


In [83]:
%%writefile MP2_func.py
from pathlib import Path
import re
import gzip
import json
data_dir = Path.home() / "Downloads"
with gzip.open(data_dir / 'movies.json.gz', 'rt', encoding='utf8') as f:
    movies = json.load(f)
with gzip.open(data_dir / 'actors.json.gz', 'rt', encoding='utf8') as f:
    actors = json.load(f)    


def search_movie(name):
    resultat = []
    for i in range(len(movies)):
        if re.search(name.lower(), (movies[i][0]).lower()):
            val = dict()
            val['name'] = movies[i][0]
            val['year'] = movies[i][2]
            val['index'] = i
            resultat.append(val)
    return resultat
        
        

def get_movie(i):
    resultat = dict()
    if len(movies) > i: 
        resultat['name'] = movies[i][0]
        resultat['year'] = movies[i][2]
        liste_acteur = []
        for indice in movies[i][1]:
            val = dict()
            val['name'] = actors[indice][0]
            val['index'] = indice
            liste_acteur.append(val)
        resultat['actor'] = liste_acteur
    return resultat
    
def search_actor(name):
    resultat = []
    for i in range(len(actors)):
        if re.search(name.lower(), (actors[i][0]).lower()):
            val = dict()
            val['name'] = actors[i][0]
            val['index'] = i
            resultat.append(val)
    return resultat
    
def get_actor(i):
    resultat = dict()
    if len(actors) > i: 
        resultat = dict()
        resultat['name'] = actors[i][0]
        liste_film = []
        for indice in actors[i][1]:
            film = {
                "name": movies[indice][0],
                "year": movies[indice][2],
                "index": indice
            }
            liste_film.append(film)
        resultat['movies'] = liste_film
    return resultat


Overwriting MP2_func.py


In [84]:
bronzés = search_movie('bronzés')
bronzés

[{'name': 'Les bronzés', 'year': 1978, 'index': 54000},
 {'name': 'Les bronzés font du ski', 'year': 1979, 'index': 55034},
 {'name': 'Les bronzés 3: amis pour la vie', 'year': 2006, 'index': 180716},
 {'name': "Les P'tits Bronzés au Pyrénéen", 'year': 2013, 'index': 467484}]

In [85]:
search_movie('gendarme')

[{'name': 'Le gendarme de Saint-Tropez', 'year': 1964, 'index': 41008},
 {'name': 'Le gendarme à New York', 'year': 1965, 'index': 42676},
 {'name': 'Le gendarme se marie', 'year': 1968, 'index': 44459},
 {'name': 'Le gendarme en balade', 'year': 1970, 'index': 46407},
 {'name': 'Le gendarme et les extra-terrestres', 'year': 1979, 'index': 55248},
 {'name': 'Le gendarme et les gendarmettes', 'year': 1982, 'index': 58345},
 {'name': 'Le gendarme de Champignol', 'year': 1959, 'index': 92831},
 {'name': 'El gendarme desconocido', 'year': 1941, 'index': 94279},
 {'name': 'El gendarme de la esquina', 'year': 1951, 'index': 116188},
 {'name': 'Sacrés gendarmes', 'year': 1980, 'index': 120030},
 {'name': "Hainburg - Je t'aime, gendarme", 'year': 2001, 'index': 145906},
 {'name': 'Le gendarme de Abobo', 'year': 2019, 'index': 320671},
 {'name': 'Le retour du gendarme de Abobo', 'year': 2025, 'index': 399616}]

In [86]:
get_movie(search_movie('Ils sont fous')[0]['index'])

{'name': 'Ils sont fous ces sorciers',
 'year': 1978,
 'actor': [{'name': 'Jean Lefebvre', 'index': 25388},
  {'name': 'Daniel Ceccaldi', 'index': 49815},
  {'name': 'Henri Guybet', 'index': 84598},
  {'name': 'Julien Guiomar', 'index': 72020},
  {'name': 'Renée Saint-Cyr', 'index': 23811},
  {'name': 'Catherine Lachens', 'index': 99392},
  {'name': 'Maitena Galli', 'index': 81810},
  {'name': 'Jean-Jacques Moreau', 'index': 96009},
  {'name': 'Michel Peyrelon', 'index': 85253},
  {'name': 'Dominique Vallée', 'index': 244191}]}

In [87]:
get_movie(bronzés[0]['index'])

{'name': 'Les bronzés',
 'year': 1978,
 'actor': [{'name': 'Josiane Balasko', 'index': 101298},
  {'name': 'Luis Rego', 'index': 84378},
  {'name': 'Marie-Anne Chazel', 'index': 103879},
  {'name': 'Michel Blanc', 'index': 99339},
  {'name': 'Martin Lamotte', 'index': 103319},
  {'name': 'Bruno Moynot', 'index': 103880},
  {'name': 'Gérard Jugnot', 'index': 98987},
  {'name': 'Michel Creton', 'index': 72838},
  {'name': 'Thierry Lhermitte', 'index': 103881},
  {'name': 'Dominique Lavanant', 'index': 103318}]}

In [88]:
harry = search_actor('Daniel Radcliffe')
harry

[{'name': 'Daniel Radcliffe', 'index': 278113}]

In [89]:
get_actor(harry[0]['index'])

{'name': 'Daniel Radcliffe',
 'movies': [{'name': 'The Tailor of Panama', 'year': 2001, 'index': 122982},
  {'name': "Harry Potter and the Sorcerer's Stone",
   'year': 2001,
   'index': 124511},
  {'name': 'Harry Potter and the Chamber of Secrets',
   'year': 2002,
   'index': 143847},
  {'name': 'Harry Potter and the Prisoner of Azkaban',
   'year': 2004,
   'index': 145912},
  {'name': 'Harry Potter and the Goblet of Fire',
   'year': 2005,
   'index': 153709},
  {'name': 'Harry Potter and the Order of the Phoenix',
   'year': 2007,
   'index': 165599},
  {'name': 'Harry Potter and the Half-Blood Prince',
   'year': 2009,
   'index': 175170},
  {'name': 'December Boys', 'year': 2007, 'index': 184184},
  {'name': 'Harry Potter and the Deathly Hallows: Part 1',
   'year': 2010,
   'index': 196949},
  {'name': 'Harry Potter and the Deathly Hallows: Part 2',
   'year': 2011,
   'index': 227084},
  {'name': 'Kill Your Darlings', 'year': 2013, 'index': 239345},
  {'name': 'The Lost City',

Write a function `movie_path(origin: int, destination: int) -> distance: int, path: list` that computes the collaboration distance between two actors. That distance is the length of the shortest path `(origin, act1, act2, ..., actX, destination)`, where `origin` and `act` played in the same movie, `act1` and `act2` played in the same movie, ... and
`actX` and `destination` played in the same movie.  In addition to the distance, the response should include one shortest path between the two actors, as a list of the form `["origin_name", "movie1_name", "act1_name", "movie2_name", ..., "destination_name"]`, where `movie1` is a movie that featured `origin` and `act1`, and so on...

In particular:
- One actor is by convention at distance 0 from herself. The return path should be `["origin_name"]` then;
- Two distinct actors that play in the same movie are at distance 1;
- If there is no connection between two actors, the function should return `-1, []` by convention.

**Important remarks**: `movie_path` is tricky. You need to try to implement it but you are allowed to fail. If you are stuck for too long, please explain what you did/try and what blocked you in your opinion. Then move on.

Your answer here.

Prompts : ChatGPT, perplexity
> Peux-tu me coder en python, une fonction movie_path qui prend en paramètre 2 arguments : origin, destination. Les 2 arguments sont des listes contenant des dictionnaires. On va regarder d'abord les films auquels chaque acteur a participé, pour cela on dispose pour chaque dictionnaire 2 éléments : "name" "index". Grâce à cet index, je peux recupèrer la liste des index des films auquel cet acteur a participé en faisant actor[index][1]. Pour récupérer le nom des films maintenant, on a juste à faire movies[index_film][0]. Si dans les 2 listes de films, les 2 acteurs n'ont pas joué ensemble alors, on va regarder pour chaque film, les acteurs qui ont joué. On va vérifier si il n'y a pas un acteur en commun. La fonction retournera une liste du genre : ["origin_name", "movie1_name", "act1_name", "movie2_name", ..., "destination_name"]. Sans utiliser de bibliothèque externe si possible.

> En fait, on n'a pas de liste en arguments mais plutot l'index de l'acteur / actrice, peux-tu modifier la fonction que tu as généré au dessus avec ce que je viens de dire ? 

> Explique moi le code comme à un étudiant en université. Commence d'abord par expliquer ce que le code fait. Ensuite, explique la logique derrière chaque boucle, condition etc... Tu peux faire un exemple pour illustrer ce que tu expliques.

> Peux-tu donner un texte qui explique tout du résumé logique ?

"La fonction movie_path permet de trouver le plus court lien entre deux acteurs en passant par les films et leurs co-acteurs.
Elle modélise la base de données comme un graphe et applique un algorithme de recherche en largeur (BFS) afin de garantir que le chemin obtenu utilise le minimum de films.

Elle évite les boucles grâce à un ensemble d’acteurs déjà visités, construit progressivement un chemin alternant acteurs et films, et s’arrête dès que l’acteur cible est atteint.
Si aucun lien n’existe, elle indique qu’aucun chemin n’a été trouvé.

Le résultat est un chemin acteur–film–acteur–… ainsi que la distance exprimée en nombre de films."

In [90]:
%%writefile -a MP2_func.py
def movie_path(origin_index, destination_index):
    """
    origin_index, destination_index : indices des acteurs dans la liste actors
    Retourne : distance (int), chemin (list)
    """

    # Cas particulier : même acteur
    if origin_index == destination_index:
        return 0, [actors[origin_index][0]]

    # BFS avec liste Python
    queue = [(origin_index, [actors[origin_index][0]])]
    visited = set()
    visited.add(origin_index)

    while queue:
        current_actor_index, path = queue.pop(0)

        # Films de l'acteur courant
        films = actors[current_actor_index][1]

        for film_index in films:
            film_name = movies[film_index][0]
            co_actors_index = movies[film_index][1]

            for co_actor_index in co_actors_index:
                if co_actor_index in visited:
                    continue

                co_actor_name = actors[co_actor_index][0]
                new_path = path + [film_name, co_actor_name]

                # Vérifier si c'est la destination
                if co_actor_index == destination_index:
                    distance = (len(new_path) - 1) // 2
                    return distance, new_path

                # Ajouter à la file
                queue.append((co_actor_index, new_path))
                visited.add(co_actor_index)

    # Aucun chemin trouvé
    return -1, []


Appending to MP2_func.py


In [91]:
def movie_path(origin_index, destination_index):
    """
    origin_index, destination_index : indices des acteurs dans la liste actors
    Retourne : distance (int), chemin (list)
    """

    # Cas particulier : même acteur
    if origin_index == destination_index:
        return 0, [actors[origin_index][0]]

    # BFS avec liste Python
    queue = [(origin_index, [actors[origin_index][0]])]
    visited = set()
    visited.add(origin_index)

    while queue:
        current_actor_index, path = queue.pop(0)

        # Films de l'acteur courant
        films = actors[current_actor_index][1]

        for film_index in films:
            film_name = movies[film_index][0]
            co_actors_index = movies[film_index][1]

            for co_actor_index in co_actors_index:
                if co_actor_index in visited:
                    continue

                co_actor_name = actors[co_actor_index][0]
                new_path = path + [film_name, co_actor_name]

                # Vérifier si c'est la destination
                if co_actor_index == destination_index:
                    distance = (len(new_path) - 1) // 2
                    return distance, new_path

                # Ajouter à la file
                queue.append((co_actor_index, new_path))
                visited.add(co_actor_index)

    # Aucun chemin trouvé
    return -1, []

In [92]:
jean = search_actor('jean dujardin')
jean_index = jean[0]['index']
jean

[{'name': 'Jean Dujardin', 'index': 330558}]

In [93]:
jack = search_actor('kiefer sutherland')
jack_index = jack[0]['index']
jack

[{'name': 'Kiefer Sutherland', 'index': 123950}]

In [94]:
kevin = search_actor('kevin bacon')
kevin_index = kevin[0]['index']
kevin

[{'name': 'Kevin Bacon', 'index': 105473},
 {'name': 'Kevin Bacon', 'index': 1210023}]

In [95]:
cruchot = search_actor('louis de funès')
cruchot_index = cruchot[0]['index']
cruchot

[{'name': 'Louis de Funès', 'index': 38583}]

In [96]:
movie_path(kevin_index, kevin_index)

(0, ['Kevin Bacon'])

In [97]:
movie_path(kevin_index, jean_index)

(2,
 ['Kevin Bacon',
  'End of the Line',
  'Bob Balaban',
  'The Monuments Men',
  'Jean Dujardin'])

In [98]:
movie_path(cruchot_index, jack_index)

(3,
 ['Louis de Funès',
  'Dernier refuge',
  'Noël Roquevert',
  'Mare matto',
  'Tomas Milian',
  'The Cowboy Way',
  'Kiefer Sutherland'])

## Exercise 6. Provide a Web API

Using Python and Flask, build a web server that implements the following routes:
- `/movies/{id}` : where `id` is the index of a movie, returns the corresponding movie as a json (cf `get_movie`).
- `/movies` : returns by default the first 100 movies. The value 100 can be modified by sending a URL parameter `limit`.
- `/actors/{id}` : where `id` is the index of an author, returns the json of the actor (cf `get_actor`).
- `/actors` : returns by default the first 100 actors. The value 100 can be modified by sending a URL parameter `limit`.
- `/actors/{id}/costars` : returns the co-stars of one actor (actors that play in a same movie).
- `/search/actors/{searchString}` : where `searchString` is a string to lookup one actor. This route should return the actors whose name contains `searchString` (for example, `/search/actors/w` returns the actors whose name contains `w` or `W`).
- `/search/movies/{searchString}`: where `searchString` is a string, returns the list of movies whose title contains `searchString`. The route should accept a URL parameter `filter` formatted like `key1:value1,key2:value2,...`  to restrain the search to the publications where key `keyi` contains `valuei`. For example, `/search/movies/gendarme?filter=year:1964`
should return the list of movies where the title contains `gendarme` published in 1964.
- `/actors/{id_origin}/distance/{id_destination}` : where `id_origin`
and `id_destination` are two actor indices, returns the collaboration distance between the two actors. In addition to the distance, the response should include one shortest path between the two actors, e.g. the json you return should be a list of two elements, one integer and one list.

The developed API should have the following characteristics:

- All errors should have the same format.
- In absence of error, the API should always return a `json`.
- Each route must be documented with the return format, possible errors, and an explanation of parameters.
- Each route that returns a list should return a maximum of 100 elements and should accept URL parameters `start` and `limit` to display `limit` elements starting from the `start`-th element. For example: `/actors` should return the first 100 authors, `/actors?start=100` displays the next 100, and `/actors?start=200&limit=2` displays the next 2 elements.
- For each route that returns a list, the returned elements should be sortable based on a given field using a URL parameter `order`. For example: `/movies?order=year` displays the first 100 movies sorted by year.

Your answer here.

Prompts: ChatGPT
> Flask n'est pas installé dans jupyter, même après avoir fait un pip install Flask.

> Comment mettre le dictionnaire obtenu à partir des fonctions en un JSON qui s'affichera dans app.route ?

> Comment puis-je retourner un JSON de manière bien structurée quand je vais sur le lien ? Il y a des caractères comme le é qui est mal affiché, il est écrit comme ça \u00e9 quand je l'affiche dans le lien

> Si on n'a pas de paramètre limit que renvoie t-il ? Comment on fait pour avoir une valeur par défaut ?

> 127.0.0.1 - - [19/Dec/2025 21:39:15] "GET /movies HTTP/1.1" 500 - ... TypeError: 'function' object is not subscriptable, que veut dire cette erreur ?

> Lorsque je rentre une URL en particulier, parfois ça tourne en boucle et ça ne veut rien afficher. Est-ce que c'est parce que ça prend trop de temps à exécuter ?

> Peux-tu me coder la fonction actors_costars(id) qui prend en paramètre l'id (int) de l'acteur et me donne tous les acteurs avec qui il a joué. Ca veut dire qu'on regarde tous les films dans lesquels l'acteur a joueé et on recupère à partir de ces films, les acteurs qui ont joué dedans.

> Comment je peux appliquer tous les filtres passés en paramètre de mon URL sur ma liste ?

> Peux-tu me coder la fonction actors_distance(id_origin, id_destination) qui prend en paramètre un identifiant de départ et un identifiant d'arrivée (les 2 sont des int). Je voudrais que tu utilises la fonction movie_path qu'on a codé (Inserer le code)

> Code moi une fonction qui me permettra de retourner les erreurs au même format. C'est a dire qu'on peut avoir l'acteur n'existe pas ou alors même l'acteur n'existe pas et d'autre erreurs etc... Je veux juste qu'il soit du même format.

> J'utilise tout le temps     
     return Response(
        json.dumps(data, ensure_ascii=False, indent=2),
        status=status,
        content_type="application/json; charset=utf-8"
    )
> Ce serait mieux si on le définissait comme une fonction non ?

> Explique moi le code comme à un étudiant en université. Commence d'abord par expliquer ce que le code fait. Ensuite, explique la logique derrière chaque boucle, condition etc... Si possible de manière concise, vu que le code est un peu long (Inserer le code)

"Le code met en place une API web avec Flask pour interagir avec une base de données de films et d’acteurs.
Les fonctionnalités principales sont :
- Récupération d’un film ou d’un acteur par leur index (/movies/<id>, /actors/<id>).
- Liste de films ou d’acteurs avec pagination et tri (/movies, /actors).
- Recherche par nom pour films et acteurs (/search/movies/<searchString>, /search/actors/<searchString>).
- Récupération des co-acteurs d’un acteur (/actors/<id>/costars).
- Calcul du chemin / distance entre deux acteurs (/actors/<id_origin>/distance/<id_destination>).
  
Les réponses sont renvoyées en JSON pour être facilement utilisées côté client ou par d’autres applications.

1. Fonctions utilitaires :
- make_json_response(data, status) : transforme n’importe quel objet Python en JSON pour la réponse HTTP.
- make_error(message, details, status) : génère un JSON uniforme pour les erreurs.

2. Récupération individuelle d’un film ou acteur :
- Les routes /movies/<id> et /actors/<id> appellent respectivement get_movie(id) et get_actor(id).
- Si l’élément n’existe pas, renvoie une erreur 404.

3. Liste paginée :
- Les routes /movies et /actors permettent de récupérer des listes avec start et limit.
- On limite le limit à 100 pour éviter de surcharger le serveur.
- Possibilité de trier via order sur un champ du dictionnaire.

4. Recherche par motif (pattern) :
- /search/movies/<searchString> et /search/actors/<searchString> utilisent les fonctions search_movie et search_actor.
- On applique la pagination et éventuellement un tri.
- Pour les films, on peut filtrer avec filter=key:value.

5. Récupération des co-acteurs :
- On parcourt tous les films de l’acteur cible.
- Pour chaque film, on ajoute à un set tous les acteurs différents de l’acteur d’origine.
- On transforme le set en liste avec pagination et tri si demandé.

6. Calcul de distance entre deux acteurs :
- La route /actors/<id_origin>/distance/<id_destination> utilise la fonction movie_path."

ChatGPT ma proposer plusieurs points que je pourrais améliorer dans mon code :
- Factoriser la pagination et tri.
- Optimiser l’accès aux films et acteurs pour les grosses bases.
- Ajouter logging et docstrings.
- Vérifier la robustesse des filtres et des types.

In [99]:
%%writefile MP2.py
from flask import Flask, Response, request
import json
from MP2_func import movies, actors, get_movie, get_actor, search_actor, search_movie, movie_path

app = Flask("MyApp")

def make_json_response(data, status=200):
    return Response(
        json.dumps(data, ensure_ascii=False, indent=2),
        status=status,
        content_type="application/json; charset=utf-8"
    )

def make_error(message, details=None, status=400):
    return make_json_response({"error": message, "details": details or {}}, status=status)

@app.route('/movies/<int:id>')
def movie(id):
    try:
        movie_info = get_movie(id)
        if not movie_info:
            return make_error("Movie not found", {"id": id}, 404)
        return make_json_response(movie_info)
    except Exception as e:
        return make_error("Internal server error", {"exception": str(e)}, 500)

@app.route('/movies')
def affiche_liste_movie():
    try:
        start = int(request.args.get("start", 0))
        limit = int(request.args.get("limit", 100))
        if limit > 100:
            limit = 100
        liste_film = []
        if len(movies) > start:
            if len(movies) > start+limit:   
                liste_film = [get_movie(i) for i in range(start,start+limit)]
            else:
                liste_film = [get_movie(i) for i in range(start,len(movies))]
        order_field = request.args.get("order")
        if order_field:
            liste_film.sort(key=lambda x: x.get(order_field))
        return make_json_response(liste_film)
    except Exception as e:
        return make_error("Internal server error", {"exception": str(e)}, 500)

@app.route('/actors/<int:id>')
def recherche_acteur(id):
    try:
        actor_info = get_actor(id)
        if not actor_info:
            return make_error("Actor not found", {"id": id}, 404)
        return make_json_response(actor_info)
    except Exception as e:
        return make_error("Internal server error", {"exception": str(e)}, 500)

@app.route('/actors')
def affiche_liste_acteur():
    try:
        start = int(request.args.get("start", 0))
        limit = int(request.args.get("limit", 100))
        if limit > 100:
            limit = 100
        liste_acteur = []
        if len(actors) > start:
            if len(actors) > start+limit:   
                liste_acteur = [get_actor(i) for i in range(start,start+limit)]
            else:
                liste_acteur = [get_actor(i) for i in range(start,len(actors))]
            order_field = request.args.get("order")
            if order_field:
                liste_acteur.sort(key=lambda x: x.get(order_field))
        return make_json_response(liste_acteur)
    except Exception as e:
        return make_error("Internal server error", {"exception": str(e)}, 500)
    
@app.route('/actors/<int:id>/costars')
def actors_costars(id):
    try: 
        actor_info = get_actor(id)
        costars = set()
        start = int(request.args.get("start", 0))
        limit = int(request.args.get("limit", 100))
        if limit > 100:
            limit = 100
        for film in actor_info["movies"]:
            movie_info = get_movie(film["index"])
            for coactor in movie_info["actor"]:
                if coactor["index"] != id:
                    costars.add((coactor["index"], coactor["name"]))
        costars_list = [{"index": idx, "name": name} for idx, name in costars]
        costars_list = costars_list[start : start + limit]
        order_field = request.args.get("order")
        if order_field:
            costars_list.sort(key=lambda x: x.get(order_field))
        return make_json_response(costars_list)
    except Exception as e:
        return make_error("Internal server error", {"exception": str(e)}, 500)

@app.route('/search/actors/<searchString>')
def recherche_acteur_pattern(searchString):
    try:
        start = int(request.args.get("start", 0))
        limit = int(request.args.get("limit", 100))
        if limit > 100:
            limit = 100
        resultat = search_actor(searchString)
        resultat = resultat[start:start+limit]
        order_field = request.args.get("order")
        if order_field:
            resultat.sort(key=lambda x: x.get(order_field))
        return make_json_response(resultat)
    except Exception as e:
        return make_error("Internal server error", {"exception": str(e)}, 500)

@app.route('/search/movies/<searchString>')
def recherche_film_pattern(searchString):
    try:
        filter_param = request.args.get("filter") 
        start = int(request.args.get("start", 0))
        limit = int(request.args.get("limit", 100))
        if limit > 100:
            limit = 100
        filters = {}
        
        if filter_param:
            for f in filter_param.split(","):
                if ":" in f:
                    key, value = f.split(":", 1)
                    filters[key] = value
        
        liste_of_movies = search_movie(searchString)
        
        results = liste_of_movies
        for key, value in filters.items():
            results = [elt for elt in results if str(elt.get(key)) == value]
    
        results = results[start:start+limit]
        order_field = request.args.get("order")
        if order_field:
            results.sort(key=lambda x: x.get(order_field))
        return make_json_response(results)
    except Exception as e:
        return make_error("Internal server error", {"exception": str(e)}, 500)

@app.route('/actors/<int:id_origin>/distance/<int:id_destination>')
def actors_distance(id_origin, id_destination):
    try:
        distance, path = movie_path(id_origin, id_destination)
        distance_to_return = distance if distance >= 0 else -1
        result_final = [distance_to_return, path]
        return make_json_response(result_final)
    except Exception as e:
        return make_error("Internal server error", {"exception": str(e)}, 500)

if __name__ == "__main__":
    app.run(host="127.0.0.1", port=5001, debug=True)


Overwriting MP2.py


## Exercise 7. Test a Web API

Using `pytest`, write a program that checks that the API made in the previous exercise works as expected.

Your answer here.

Prompt: ChatGPT
> Comment je fais pour tester mon API avec pytest ? Parce que d'après ce que j'ai vu on peut le faire que si on a un fichier .py

> Comment je fais alors pour executer mon Flask pour les tests ?

> Génère moi un code qui permet de tester tous mes app.route et de vérifier que j'ai bien respecté les règles qui sont : d'avoir les erreurs au même format, en abscence d'erreur retournée un JSON et pour chaque liste retournée les 100 premiers éléments sauf si on a les paramètres limit et start ainsi que ORDER qui trie la liste.

> A quoi servent les couleurs ANSI ?

> Peux-tu m'expliquer tout ce que teste le code actuel ?

In [100]:
%%writefile test_MP2.py
import json
import pytest

from MP2 import app


@pytest.fixture
def client():
    app.testing = True
    with app.test_client() as client:
        yield client


# ---------- /movies/{id} ----------

def test_get_movie_ok(client):
    resp = client.get("/movies/0")
    assert resp.status_code == 200
    data = resp.get_json()
    assert isinstance(data, dict)
    assert "name" in data
    assert "year" in data
    assert "actor" in data
    assert isinstance(data["actor"], list)


def test_get_movie_not_found(client):
    resp = client.get("/movies/99999999")
    assert resp.status_code == 404
    data = resp.get_json()
    assert isinstance(data, dict)
    assert "error" in data
    assert data["error"] == "Movie not found"


# ---------- /movies (liste) ----------

def test_movies_default_limit(client):
    resp = client.get("/movies")
    assert resp.status_code == 200
    data = resp.get_json()
    assert isinstance(data, list)
    assert len(data) <= 100


def test_movies_limit_and_start(client):
    resp1 = client.get("/movies?start=0&limit=5")
    resp2 = client.get("/movies?start=5&limit=5")
    assert resp1.status_code == 200
    assert resp2.status_code == 200
    list1 = resp1.get_json()
    list2 = resp2.get_json()
    assert len(list1) <= 5
    assert len(list2) <= 5
    # si au moins 5 films existent, les blocs ne doivent pas être identiques
    if len(list1) == 5 and len(list2) == 5:
        assert list1 != list2


def test_movies_order_by_year(client):
    resp = client.get("/movies?order=year&limit=20")
    assert resp.status_code == 200
    data = resp.get_json()
    years = [m.get("year") for m in data if "year" in m]
    assert years == sorted(years)


# ---------- /actors/{id} ----------

def test_get_actor_ok(client):
    resp = client.get("/actors/0")
    assert resp.status_code == 200
    data = resp.get_json()
    assert isinstance(data, dict)
    assert "name" in data
    assert "movies" in data
    assert isinstance(data["movies"], list)


def test_get_actor_not_found(client):
    resp = client.get("/actors/99999999")
    assert resp.status_code == 404
    data = resp.get_json()
    assert isinstance(data, dict)
    assert "error" in data
    assert data["error"] == "Actor not found"


# ---------- /actors (liste) ----------

def test_actors_default_limit(client):
    resp = client.get("/actors")
    assert resp.status_code == 200
    data = resp.get_json()
    assert isinstance(data, list)
    assert len(data) <= 100


def test_actors_limit_and_start(client):
    resp1 = client.get("/actors?start=0&limit=5")
    resp2 = client.get("/actors?start=5&limit=5")
    assert resp1.status_code == 200
    assert resp2.status_code == 200
    list1 = resp1.get_json()
    list2 = resp2.get_json()
    assert len(list1) <= 5
    assert len(list2) <= 5
    if len(list1) == 5 and len(list2) == 5:
        assert list1 != list2


def test_actors_order_by_name(client):
    resp = client.get("/actors?order=name&limit=20")
    assert resp.status_code == 200
    data = resp.get_json()
    names = [a.get("name") for a in data if "name" in a]
    assert names == sorted(names)


# ---------- /actors/{id}/costars ----------

def test_costars_route(client):
    resp = client.get("/actors/0/costars")
    assert resp.status_code == 200
    data = resp.get_json()
    assert isinstance(data, list)
    for co in data:
        assert "name" in co
        assert "index" in co


# ---------- /search/actors/{searchString} ----------

def test_search_actors(client):
    resp = client.get("/search/actors/jean")
    assert resp.status_code == 200
    data = resp.get_json()
    assert isinstance(data, list)
    # On teste juste que ça renvoie bien une liste d'objets avec au moins name / index si présent
    for actor in data:
        assert "name" in actor
        assert "index" in actor


# ---------- /search/movies/{searchString} + filter ----------

def test_search_movies_with_filter(client):
    resp = client.get("/search/movies/gendarme?filter=year:1964")
    assert resp.status_code == 200
    data = resp.get_json()
    assert isinstance(data, list)
    for movie in data:
        assert "name" in movie
        assert "year" in movie
        # Vérifie que le filtre année est bien appliqué
        assert movie["year"] == 1964


# ---------- /actors/{id_origin}/distance/{id_destination} ----------

def test_distance_route_same_actor(client):
    resp = client.get("/actors/0/distance/0")
    assert resp.status_code == 200
    data = resp.get_json()
    assert isinstance(data, list)
    assert len(data) == 2
    distance, path = data
    assert distance == 0
    assert isinstance(path, list)
    assert len(path) >= 1  # au moins le nom de l'acteur


def test_distance_route_two_actors(client):
    # on vérifie juste que l'API répond et renvoie bien [int, list]
    resp = client.get("/actors/0/distance/1")
    assert resp.status_code == 200
    data = resp.get_json()
    assert isinstance(data, list)
    assert len(data) == 2
    distance, path = data
    assert isinstance(distance, int)
    assert isinstance(path, list)



Overwriting test_MP2.py


In [101]:
!pytest test_MP2.py -v

============================= test session starts =============================
platform win32 -- Python 3.12.3, pytest-9.0.2, pluggy-1.6.0 -- C:\Users\inese\AppData\Local\Programs\Python\Python312\python.exe
cachedir: .pytest_cache
rootdir: c:\Users\inese\Bureau\SU\master\s1\PROGRES\MP2
plugins: anyio-4.8.0
collecting ... collected 15 items

test_MP2.py::test_get_movie_ok PASSED                                    [  6%]
test_MP2.py::test_get_movie_not_found PASSED                             [ 13%]
test_MP2.py::test_movies_default_limit PASSED                            [ 20%]
test_MP2.py::test_movies_limit_and_start PASSED                          [ 26%]
test_MP2.py::test_movies_order_by_year PASSED                            [ 33%]
test_MP2.py::test_get_actor_ok PASSED                                    [ 40%]
test_MP2.py::test_get_actor_not_found PASSED                             [ 46%]
test_MP2.py::test_actors_default_limit PASSED                            [ 53%]
test_MP2.py::te

## Exercise 8. Make a Website that uses the Web API

Create a Python web server using Flask. Use the Web API you developed to offer the user a graphical Web interface. This interface allows the user to obtain, by entering relevant information into a Web form:

- The complete list of movies and the complete list of costars of an actor, possibly sorted alphabetically. This actor can be searched beforehand using a substring of characters appearing in her name.
- The colloration distance between two actors. As above, the actors can be searched beforehand using a substring of characters appearing in their names. Try to format a bit (not too much). For example:
  - The collaboration distance between Kevin Bacon and Jean Dujardin is 2.
  - Kevin bacon played in Wild things with Bill Murray;
  - Bill Murray played in The Monuments Men with Jean Dujardin.

Your answer here.

Prompt : ChatGPT
> (Insérer le code) Génère moi une interface Web qui utilise mon API MyAPP que je t'ai insérer au dessus. L'interface comportera 2 zones, la première partie on cherchera un acteur, puis a partir de cette auteur je veux les films et tout les acteurs qui ont jouer avec cette acteur. La 2ème partie, on cherchera la distance entre 2 acteur, c-a-d qu'on utilisera actors distance de mon API

> Pour les films sa contient name year et actor, ou actor contient nom + index Alors que les acteurs, contient name et movies, ou movies contient name year index

> J'ai des erreurs ou le port est déjà utilisées comment je fais ?

> File "/Users/loic/Downloads/WebIU.py", line 182, in actor_search actors_list = [SimpleActor(a["name"], a["id"]) for a in raw] ~^^^^^^ KeyError: 'id C'est quoi cette erreur ?

> Quand je met un "string" je veux avoir tout les acteurs qui ont ce string dans leur name, du coup je veux pouvoir selectionner l'acteur que je veux

> Comment je fais pour lancer mes 2 API en même temps ? Parce que d'après ce que tu m'a dis je dois devoir lancer d'abord mon API MyAPP puis lancer WebUI

> Explique moi ce que fait mon code, de façon a ce que un étudiant comprenne.

"Ce script crée une interface web en utilisant Flask. Il permet de rechercher des acteurs, de voir les films dans lesquels ils ont joué et leurs co-stars, ainsi que de calculer la distance de collaboration entre deux acteurs. L’interface est entièrement sur une seule page HTML, avec des formulaires dynamiques pour saisir des noms d’acteurs et choisir les options d’affichage. Les données proviennent de ton module MP2_func, et sont transformées en objets simples SimpleActor et SimpleMovie pour faciliter l’affichage. Les fonctions de recherche et de calcul de distance sont appelées côté serveur, et les résultats sont affichés directement dans le navigateur grâce au moteur de template Jinja2. L’application s’exécute sur le port 5002 et n’utilise pas le mode debug pour éviter les problèmes dans Jupyter.

Le code lance ton API Flask et ton WebUI Flask simultanément dans la même cellule en utilisant des threads séparés, chacun sur un port différent.
Le time.sleep(2) assure que l’API est prête avant que le WebUI commence à l’utiliser."

Pour tester l'interface graphique il faudra aller sur http://127.0.0.1:5002

In [102]:
%%writefile WebIU.py
from flask import Flask, request, render_template_string
import requests

app = Flask("WebUI")

API_URL = "http://127.0.0.1:5001"  # ton API

# ----------------- Templates HTML -----------------
BASE_TEMPLATE = """
<!doctype html>
<html lang="en">
<head>
  <meta charset="utf-8">
  <title>IMDB Mini-projet</title>
  <style>
    body { font-family: Arial, sans-serif; margin: 20px; }
    h1 { color: #333; }
    form { margin-bottom: 20px; padding: 10px; border: 1px solid #ddd; }
    label { display: inline-block; width: 200px; }
    select, input[type="text"] { width: 300px; }
    ul { list-style-type: disc; margin-left: 20px; }
    .section { margin-bottom: 40px; }
    .result { padding: 10px; border: 1px solid #ccc; background: #f9f9f9; }
  </style>
</head>
<body>
<h1>IMDB – Interface Web</h1>

<div class="section">
<h2>1. Films et co-stars d’un acteur</h2>
<form method="get" action="{{ url_for('actor_search') }}">
  <label for="pattern">Recherche acteur :</label>
  <input type="text" name="pattern" value="{{ pattern or '' }}" required>
  <button type="submit">Chercher</button>
</form>

{% if actors_list %}
<form method="get" action="{{ url_for('actor_details') }}">
  <input type="hidden" name="pattern" value="{{ pattern }}">
  <label for="actor_id">Acteur trouvé :</label>
  <select name="actor_id">
    {% for a in actors_list %}
      <option value="{{ a['index'] }}">{{ a['name'] }} (id={{ a['index'] }})</option>
    {% endfor %}
  </select>
  <button type="submit">Afficher films & co-stars</button>
</form>
{% endif %}

{% if selected_actor %}
<div class="result">
<h3>Acteur : {{ selected_actor['name'] }} (id={{ selected_actor['index'] }})</h3>
<h4>Films :</h4>
<ul>
{% for m in movies_list %}
  <li>{{ m['name'] }} ({{ m['year'] }})</li>
{% endfor %}
</ul>
<h4>Co-stars :</h4>
<ul>
{% for c in costars_list %}
  <li>{{ c['name'] }} (id={{ c['index'] }})</li>
{% endfor %}
</ul>
</div>
{% endif %}
</div>

<hr>

<div class="section">
<h2>2. Distance de collaboration entre deux acteurs</h2>
<form method="get" action="{{ url_for('distance_search') }}">
  <label>Acteur 1 :</label>
  <input type="text" name="pattern1" value="{{ pattern1 or '' }}" required><br>
  <label>Acteur 2 :</label>
  <input type="text" name="pattern2" value="{{ pattern2 or '' }}" required><br>
  <button type="submit">Chercher</button>
</form>

{% if candidates1 or candidates2 %}
<form method="get" action="{{ url_for('distance_result') }}">
  <input type="hidden" name="pattern1" value="{{ pattern1 }}">
  <input type="hidden" name="pattern2" value="{{ pattern2 }}">
  {% if candidates1 %}
    <label>Acteur 1 :</label>
    <select name="actor1_id">
    {% for a in candidates1 %}
      <option value="{{ a['index'] }}">{{ a['name'] }} (id={{ a['index'] }})</option>
    {% endfor %}
    </select><br>
  {% endif %}
  {% if candidates2 %}
    <label>Acteur 2 :</label>
    <select name="actor2_id">
    {% for a in candidates2 %}
      <option value="{{ a['index'] }}">{{ a['name'] }} (id={{ a['index'] }})</option>
    {% endfor %}
    </select><br>
  {% endif %}
  <button type="submit">Calculer distance</button>
</form>
{% endif %}

{% if distance is not none %}
<div class="result">
{% if distance >= 0 %}
<p>Distance entre <strong>{{ name1 }}</strong> et <strong>{{ name2 }}</strong> : {{ distance }}</p>
{% if path %}
<ul>
{% for step in path %}
<li>{{ step }}</li>
{% endfor %}
</ul>
{% endif %}
{% else %}
<p>Aucun chemin trouvé.</p>
{% endif %}
</div>
{% endif %}
</div>
</body>
</html>
"""

def format_path(path):
    # transforme [actor1, movie1, actor2...] en phrases lisibles
    if not path or len(path) < 3:
        return []
    formatted = []
    for i in range(0, len(path)-2, 2):
        a1, movie, a2 = path[i], path[i+1], path[i+2]
        formatted.append(f"{a1} joué dans {movie} avec {a2}")
    return formatted

# ----------------- Routes -----------------

@app.route("/")
def index():
    return render_template_string(BASE_TEMPLATE,
                                  pattern=None, actors_list=None,
                                  selected_actor=None, movies_list=None,
                                  costars_list=None, pattern1=None,
                                  pattern2=None, candidates1=None,
                                  candidates2=None, distance=None,
                                  name1=None, name2=None, path=None)

@app.route("/actor/search")
def actor_search():
    pattern = request.args.get("pattern", "").strip()
    actors_list = []
    if pattern:
        r = requests.get(f"{API_URL}/search/actors/{pattern}")
        if r.ok:
            actors_list = r.json()
    return render_template_string(BASE_TEMPLATE,
                                  pattern=pattern, actors_list=actors_list,
                                  selected_actor=None, movies_list=None,
                                  costars_list=None, pattern1=None,
                                  pattern2=None, candidates1=None,
                                  candidates2=None, distance=None,
                                  name1=None, name2=None, path=None)

@app.route("/actor/details")
def actor_details():
    actor_id = request.args.get("actor_id", type=int)
    pattern = request.args.get("pattern", "")
    selected_actor = None
    movies_list = []
    costars_list = []
    actors_list = []

    # récupération liste pour le select
    if pattern:
        r = requests.get(f"{API_URL}/search/actors/{pattern}")
        if r.ok:
            actors_list = r.json()

    if actor_id is not None:
        r = requests.get(f"{API_URL}/actors/{actor_id}")
        if r.ok:
            info = r.json()
            selected_actor = {"name": info["name"], "index": actor_id}
            movies_list = info["movies"]
            # récupération co-stars
            r2 = requests.get(f"{API_URL}/actors/{actor_id}/costars")
            if r2.ok:
                costars_list = r2.json()

    return render_template_string(BASE_TEMPLATE,
                                  pattern=pattern, actors_list=actors_list,
                                  selected_actor=selected_actor, movies_list=movies_list,
                                  costars_list=costars_list, pattern1=None,
                                  pattern2=None, candidates1=None,
                                  candidates2=None, distance=None,
                                  name1=None, name2=None, path=None)

@app.route("/distance/search")
def distance_search():
    pattern1 = request.args.get("pattern1", "")
    pattern2 = request.args.get("pattern2", "")
    candidates1, candidates2 = [], []
    if pattern1:
        r = requests.get(f"{API_URL}/search/actors/{pattern1}")
        if r.ok:
            candidates1 = r.json()
    if pattern2:
        r = requests.get(f"{API_URL}/search/actors/{pattern2}")
        if r.ok:
            candidates2 = r.json()
    return render_template_string(BASE_TEMPLATE,
                                  pattern=None, actors_list=None,
                                  selected_actor=None, movies_list=None,
                                  costars_list=None, pattern1=pattern1,
                                  pattern2=pattern2, candidates1=candidates1,
                                  candidates2=candidates2, distance=None,
                                  name1=None, name2=None, path=None)

@app.route("/distance/result")
def distance_result():
    actor1_id = request.args.get("actor1_id", type=int)
    actor2_id = request.args.get("actor2_id", type=int)
    pattern1 = request.args.get("pattern1", "")
    pattern2 = request.args.get("pattern2", "")
    distance = None
    path = []
    name1 = name2 = None
    candidates1 = candidates2 = []

    # récupérer candidats pour le select
    if pattern1:
        r = requests.get(f"{API_URL}/search/actors/{pattern1}")
        if r.ok:
            candidates1 = r.json()
    if pattern2:
        r = requests.get(f"{API_URL}/search/actors/{pattern2}")
        if r.ok:
            candidates2 = r.json()

    if actor1_id is not None and actor2_id is not None:
        r = requests.get(f"{API_URL}/actors/{actor1_id}/distance/{actor2_id}")
        if r.ok:
            res = r.json()
            distance = res[0]
            path = format_path(res[1])
        # récupérer noms pour affichage
        r1 = requests.get(f"{API_URL}/actors/{actor1_id}")
        r2 = requests.get(f"{API_URL}/actors/{actor2_id}")
        if r1.ok and r2.ok:
            name1 = r1.json()["name"]
            name2 = r2.json()["name"]

    return render_template_string(BASE_TEMPLATE,
                                  pattern=None, actors_list=None,
                                  selected_actor=None, movies_list=None,
                                  costars_list=None,
                                  pattern1=pattern1, pattern2=pattern2,
                                  candidates1=candidates1, candidates2=candidates2,
                                  distance=distance, name1=name1, name2=name2, path=path)

if __name__ == "__main__":
    app.run(host="127.0.0.1", port=5002, debug=False)


Overwriting WebIU.py


In [ ]:
import threading
import time
from MP2 import app as api_app  # ton API Flask
from WebIU import app as webui_app  # ton WebUI Flask

def run_api():
    print("Démarrage de l'API sur le port 5001...")
    api_app.run(host="127.0.0.1", port=5001, debug=False, use_reloader=False)

def run_webui():
    print("Démarrage du WebUI sur le port 5002...")
    webui_app.run(host="127.0.0.1", port=5002, debug=False, use_reloader=False)

# Lancer l'API dans un thread
thread_api = threading.Thread(target=run_api, daemon=True)
thread_api.start()

# Petit délai pour s'assurer que l'API est prête
time.sleep(2)

# Lancer WebUI dans un thread
thread_webui = threading.Thread(target=run_webui, daemon=True)
thread_webui.start()


Démarrage de l'API sur le port 5001...
 * Serving Flask app 'MyApp'
 * Debug mode: off


 * Running on http://127.0.0.1:5001
Press CTRL+C to quit


Démarrage du WebUI sur le port 5002...


 * Serving Flask app 'WebUI'
 * Debug mode: off


 * Running on http://127.0.0.1:5002
Press CTRL+C to quit
